### K means with 2 clusters

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
from scipy.stats import norm
from sklearn.cluster import KMeans, DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, fbeta_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.covariance import EllipticEnvelope
from sklearn.mixture import BayesianGaussianMixture

c:\Users\danit\Personal Projects\dsp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../dataset/creditcard.csv')
unlabelled_df = df.drop(columns=['Class', 'Time'])
labels = df['Class']

print(labels.value_counts())

train_X, test_X, train_y, test_y = train_test_split(unlabelled_df, labels, test_size=0.35, random_state=67)
'''
scaler = StandardScaler()
train_X_scaled = scaler.fit_transform(train_X)
test_X_scaled = scaler.transform(test_X)

kmeans = KMeans(n_clusters=2, random_state=0).fit(train_X_scaled)
preds = kmeans.predict(test_X_scaled)

ss = silhouette_score(test_X_scaled, preds)
print("Silhouette Score: ", ss)
'''

Class
0    284315
1       492
Name: count, dtype: int64


'\nscaler = StandardScaler()\ntrain_X_scaled = scaler.fit_transform(train_X)\ntest_X_scaled = scaler.transform(test_X)\n\nkmeans = KMeans(n_clusters=2, random_state=0).fit(train_X_scaled)\npreds = kmeans.predict(test_X_scaled)\n\nss = silhouette_score(test_X_scaled, preds)\nprint("Silhouette Score: ", ss)\n'

In [55]:
comparison_df = pd.DataFrame({"True label": test_y, "Pred label": preds})
crosstab_res = pd.crosstab(comparison_df['True label'], comparison_df['Pred label'], rownames=['True'], colnames=['Predicted'])

print(crosstab_res)

ValueError: array length 142404 does not match index length 99683

### DBSCAN

In [ ]:
'''
epsilon = 0.3

dbscan = DBSCAN(eps=epsilon, min_samples=5)
labels = dbscan.fit_predict(train_X_scaled) 
#chs = calinski_harabasz_score(train_X_scaled, dbscan.labels_)
#dbs = davies_bouldin_score(train_X_scaled, dbscan.labels_)
#calinski = chs
#davies = dbs

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)


if n_clusters > 1:
    std_score = silhouette_score(train_X_scaled, labels)
    print("Standard Silhouette Score (with noise): ", std_score)
    mask = labels != -1
    if len(set(labels[mask])) > 1:
        pure_score = silhouette_score(train_X_scaled[mask], labels[mask])
        print("Pure Silhouette Score (without noise): ", pure_score)
'''



'\nepsilon = 0.3\n\ndbscan = DBSCAN(eps=epsilon, min_samples=5)\nlabels = dbscan.fit_predict(train_X_scaled) \n#chs = calinski_harabasz_score(train_X_scaled, dbscan.labels_)\n#dbs = davies_bouldin_score(train_X_scaled, dbscan.labels_)\n#calinski = chs\n#davies = dbs\n\nn_clusters = len(set(labels)) - (1 if -1 in labels else 0)\n\n\nif n_clusters > 1:\n    std_score = silhouette_score(train_X_scaled, labels)\n    print("Standard Silhouette Score (with noise): ", std_score)\n    mask = labels != -1\n    if len(set(labels[mask])) > 1:\n        pure_score = silhouette_score(train_X_scaled[mask], labels[mask])\n        print("Pure Silhouette Score (without noise): ", pure_score)\n'

### Isolation forest (iForest)

In [ ]:
n_estimators = 100

def objective(trial):
    max_samples = trial.suggest_categorical("max_samples", [128, 256, 512, 768, 1024])
    contamination = trial.suggest_float("contamination", 0.001, 0.01, log=True)
    iso_forest = IsolationForest(n_estimators=n_estimators, contamination=contamination, max_samples=max_samples, random_state=67)
    iso_forest.fit(train_X)
    y_pred = iso_forest.predict(test_X)
    y_pred_mapped = np.where(y_pred == -1, 1, 0)
    f2 = fbeta_score(test_y, y_pred_mapped, beta=2)
    f3 = fbeta_score(test_y, y_pred_mapped, beta=3)

    return f3

study = optuna.create_study(direction="maximize", study_name="Isolation Forest Hyperparameter Optimization")
study.optimize(objective, n_trials=50)

print("Best hyperparameters: ", study.best_params)
print("Best F2 score: ", study.best_value)
print("Best f3 score: ", study.best_value)  

[I 2026-09-17 19:05:50,644] A new study created in memory with name: Isolation Forest Hyperparameter Optimization
[I 2026-09-17 19:05:52,862] Trial 0 finished with value: 0.3372591006423983 and parameters: {'max_samples': 1024, 'contamination': 0.0028710772495270003}. Best is trial 0 with value: 0.3372591006423983.
[I 2026-09-17 19:05:54,854] Trial 1 finished with value: 0.4309303262182843 and parameters: {'max_samples': 512, 'contamination': 0.008787760493950211}. Best is trial 1 with value: 0.4309303262182843.
[I 2026-09-17 19:05:57,306] Trial 2 finished with value: 0.28048082427017745 and parameters: {'max_samples': 1024, 'contamination': 0.0017319468013987762}. Best is trial 1 with value: 0.4309303262182843.
[I 2026-09-17 19:05:59,986] Trial 3 finished with value: 0.4223042230422304 and parameters: {'max_samples': 1024, 'contamination': 0.008213814766787644}. Best is trial 1 with value: 0.4309303262182843.
[I 2026-09-17 19:06:02,127] Trial 4 finished with value: 0.3699843668577384 

Best hyperparameters:  {'max_samples': 768, 'contamination': 0.00634854434241205}
Best F2 score:  0.4725813642443156
Best f3 score:  0.4725813642443156


In [ ]:
best_model = IsolationForest(
    n_estimators=n_estimators,
    **study.best_params,
    random_state=67
)

best_model.fit(train_X)

results = pd.DataFrame(test_X, index=test_y.index)

results["anomaly_score"] = best_model.decision_function(test_X)
results["anomaly_label"] = best_model.predict(test_X)
results["true_label"] = test_y

results["anomaly_label"] = results["anomaly_label"].map({1: 0, -1: 1})
confusion_matrix = pd.crosstab(results["true_label"], results["anomaly_label"], rownames=["True"], colnames=["Predicted"])

recall = confusion_matrix.loc[1, 1] / (confusion_matrix.loc[1, 1] + confusion_matrix.loc[1, 0])
precision = confusion_matrix.loc[1, 1] / (confusion_matrix.loc[1, 1] + confusion_matrix.loc[0, 1])
f2_score = (5 * precision * recall) / (4 * precision + recall)
f3_score = (10 * precision * recall) / (9 * precision + recall)

print("Confusion Matrix:\n", confusion_matrix)
print("Recall:", recall)
print("Precision:", precision)
print("F2 Score:", f2_score)
print("F3 Score:", f3_score)

Confusion Matrix:
 Predicted      0    1
True                 
0          98946  562
1             69  106
Recall: 0.6057142857142858
Precision: 0.15868263473053892
F2 Score: 0.38742690058479534
F3 Score: 0.4725813642443157


### Bayesian Gaussian Mixture 

In [ ]:
gaussian_dataset = unlabelled_df.copy()
gaussian_dataset = gaussian_dataset.drop(columns=["V1", "V4", "V6", "V10", "V22", "V24", "V25", "V26", "Amount"])
gaussian_X_train, gaussian_X_test, gaussian_y_train, gaussian_y_test = train_test_split(gaussian_dataset, labels, test_size=0.35, random_state=67)

scaler = StandardScaler()

gaussian_X_train_scaled = scaler.fit_transform(gaussian_X_train)
gaussian_X_test_scaled = scaler.fit_transform(gaussian_X_test)

bgm = BayesianGaussianMixture(
    n_components=15, 
    covariance_type='full', 
    random_state=67,
    weight_concentration_prior_type='dirichlet_process', 
    weight_concentration_prior=0.01,
    max_iter=1000,
    n_init=5
)

bgm.fit(gaussian_X_train_scaled)
bgm_preds = bgm.predict(gaussian_X_test_scaled)
bgm


,"n_components n_components: int, default=1The number of mixture components. Depending on the data and the valueof the `weight_concentration_prior` the model can decide to not useall the components by setting some component `weights_` to values veryclose to zero. The number of effective components is therefore smallerthan n_components.",15
,"max_iter max_iter: int, default=100The number of EM iterations to perform.",1000
,"n_init n_init: int, default=1The number of initializations to perform. The result with the highestlower bound value on the likelihood is kept.",5
,"weight_concentration_prior weight_concentration_prior: float or None, default=NoneThe dirichlet concentration of each component on the weightdistribution (Dirichlet). This is commonly called gamma in theliterature. The higher concentration puts more mass inthe center and will lead to more components being active, while a lowerconcentration parameter will lead to more mass at the edge of themixture weights simplex. The value of the parameter must be greaterthan 0. If it is None, it's set to ``1. / n_components``.",0.01
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to the method chosen to initialize theparameters (see `init_params`).In addition, it controls the generation of random samples from thefitted distribution (see the method `sample`).Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",67
,"covariance_type covariance_type: {'full', 'tied', 'diag', 'spherical'}, default='full'String describing the type of covariance parameters to use.Must be one of:- 'full' (each component has its own general covariance matrix),- 'tied' (all components share the same general covariance matrix),- 'diag' (each component has its own diagonal covariance matrix),- 'spherical' (each component has its own single variance).",'full'
,"tol tol: float, default=1e-3The convergence threshold. EM iterations will stop when thelower bound average gain on the likelihood (of the training data withrespect to the model) is below this threshold.",0.001
,"reg_covar reg_covar: float, default=1e-6Non-negative regularization added to the diagonal of covariance.Allows to assure that the covariance matrices are all positive.",1e-06
,"init_params init_params: {'kmeans', 'k-means++', 'random', 'random_from_data'}, default='kmeans'The method used to initialize the weights, the means and thecovariances. String must be one of:- 'kmeans': responsibilities are initialized using kmeans.- 'k-means++': use the k-means++ method to initialize.- 'random': responsibilities are initialized randomly.- 'random_from_data': initial means are randomly selected data points... versionchanged:: v1.1 `init_params` now accepts 'random_from_data' and 'k-means++' as initialization methods.",'kmeans'
,"weight_concentration_prior_type weight_concentration_prior_type: {'dirichlet_process', 'dirichlet_distribution'}, default='dirichlet_process'String describing the type of the weight concentration prior.",'dirichlet_process'
,"mean_precision_prior mean_precision_prior: float or None, default=NoneThe precision prior on the mean distribution (Gaussian).Controls the extent of where means can be placed. Largervalues concentrate the cluster means around `mean_prior`.The value of the parameter must be greater than 0.If it is None, it is set to 1.",None


In [ ]:
weights = bgm.weights_
results = pd.DataFrame()
results['Cluster'] = bgm.predict(gaussian_X_train_scaled)
results['True_Class'] = gaussian_y_train.values 

crosstab = pd.crosstab(results['Cluster'], results['True_Class'])

crosstab = crosstab.rename(columns={0: 'Normal', 1: 'Fraud'})

if 'Fraud' not in crosstab.columns: crosstab['Fraud'] = 0
if 'Normal' not in crosstab.columns: crosstab['Normal'] = 0

crosstab['Fraud_Rate (%)'] = (crosstab['Fraud'] / (crosstab['Normal'] + crosstab['Fraud'])) * 100

print(crosstab.sort_values(by='Fraud_Rate (%)', ascending=False))


[0.03296612 0.10240141 0.24286175 0.01541777 0.04331624 0.16516981
 0.07269033 0.02272653 0.05125229 0.03813048 0.05273267 0.03405615
 0.02858528 0.04113075 0.05656243]
True_Class  Normal  Fraud  Fraud_Rate (%)
Cluster                                  
3             2577    264        9.292503
4             8005     13        0.162135
10            9710     14        0.143974
12            5262      6        0.113895
11            6303      2        0.031721
6            13484      4        0.029656
1            18870      5        0.026490
8             9486      2        0.021079
13            7614      1        0.013132
2            44956      4        0.008897
5            30697      2        0.006515
0             6103      0        0.000000
7             4213      0        0.000000
9             7059      0        0.000000
14           10468      0        0.000000


In [16]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(bgm, f)
    